In [ ]:
# 04. 구매 세션 vs 비구매 세션 행동 비교
# 한 번의 쇼핑 방문 전체를 기준으로 구매 여부에 따른 행동 차이를 분석

In [1]:
import duckdb

parquet_path = r"..\data\processed\2019-*.parquet"

In [3]:
# 세션별 행동량, 조회 상품 수, 세션 길이, 구매 여부를 하나의 테이블로 집계
# 각 세션을 한 행으로 만들어 조회·장바구니·구매 행동량과 세션 길이, 구매 여부를 정리
session_behavior = duckdb.sql(f"""
    SELECT
        user_session,

        COUNT(*) AS total_events,

        SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,
        SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_events,
        SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events,

        COUNT(DISTINCT product_id) AS unique_products,

        DATE_DIFF(
            'second',
            MIN(event_time),
            MAX(event_time)
        ) AS session_seconds,

        CASE
            WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
            THEN 1
            ELSE 0
        END AS purchased

    FROM read_parquet('{parquet_path}')

    WHERE user_session IS NOT NULL
      AND CAST(event_time AS DATE) <> '2019-11-15'

    GROUP BY user_session
""")

session_behavior.limit(10).show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────────────────┬──────────────┬─────────────┬─────────────┬─────────────────┬─────────────────┬─────────────────┬───────────┐
│             user_session             │ total_events │ view_events │ cart_events │ purchase_events │ unique_products │ session_seconds │ purchased │
│               varchar                │    int64     │   int128    │   int128    │     int128      │      int64      │      int64      │   int32   │
├──────────────────────────────────────┼──────────────┼─────────────┼─────────────┼─────────────────┼─────────────────┼─────────────────┼───────────┤
│ 3141d02d-e54d-40f6-9fa1-cdfb798ab478 │           13 │          13 │           0 │               0 │              11 │             403 │         0 │
│ 3c2576f6-9253-4314-94f1-33258a3a779a │           28 │          28 │           0 │               0 │              27 │            1925 │         0 │
│ b0ecec5f-0135-484c-8ed1-b19d03e76ec8 │            2 │           2 │           0 │               0 

In [5]:
# 구매 세션과 비구매 세션이 각각 몇 개인지 확인
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,
            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 'purchase_session'
                ELSE 'non_purchase_session'
            END AS session_type
        FROM read_parquet('{parquet_path}')
        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'
        GROUP BY user_session
    )

    SELECT
        session_type,
        COUNT(*) AS session_count
    FROM session_behavior
    GROUP BY session_type
    ORDER BY session_count DESC
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬───────────────┐
│     session_type     │ session_count │
│       varchar        │     int64     │
├──────────────────────┼───────────────┤
│ non_purchase_session │      20850198 │
│ purchase_session     │       1402758 │
└──────────────────────┴───────────────┘



In [7]:
# 확인 결과
# - 비구매 세션 20,850,198개, 구매 세션 1,402,758개
# - 전체 세션 중 구매 세션은 소수이므로 두 그룹의 행동 차이를 비교할 필요가 있음

In [9]:
# 구매 세션과 비구매 세션의 평균 행동량과 세션 길이 비교
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,

            COUNT(*) AS total_events,

            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,
            SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS cart_events,
            SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchase_events,

            COUNT(DISTINCT product_id) AS unique_products,

            DATE_DIFF(
                'second',
                MIN(event_time),
                MAX(event_time)
            ) AS session_seconds,

            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 'purchase_session'
                ELSE 'non_purchase_session'
            END AS session_type

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session
    )

    SELECT
        session_type,
        COUNT(*) AS sessions,

        ROUND(AVG(total_events), 2) AS avg_total_events,
        ROUND(AVG(view_events), 2) AS avg_views,
        ROUND(AVG(cart_events), 2) AS avg_carts,
        ROUND(AVG(unique_products), 2) AS avg_unique_products,

        ROUND(AVG(session_seconds), 2) AS avg_session_seconds,
        MEDIAN(session_seconds) AS median_session_seconds

    FROM session_behavior

    GROUP BY session_type
    ORDER BY session_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────┬──────────┬──────────────────┬───────────┬───────────┬─────────────────────┬─────────────────────┬────────────────────────┐
│     session_type     │ sessions │ avg_total_events │ avg_views │ avg_carts │ avg_unique_products │ avg_session_seconds │ median_session_seconds │
│       varchar        │  int64   │      double      │  double   │  double   │       double        │       double        │         double         │
├──────────────────────┼──────────┼──────────────────┼───────────┼───────────┼─────────────────────┼─────────────────────┼────────────────────────┤
│ non_purchase_session │ 20850198 │             4.42 │      4.34 │      0.09 │                2.97 │             1134.55 │                   46.0 │
│ purchase_session     │  1402758 │             8.19 │      5.81 │      1.19 │                3.01 │              706.04 │                  257.0 │
└──────────────────────┴──────────┴──────────────────┴───────────┴───────────┴─────────────────────┴────────────

In [11]:
# 확인 결과
# - 구매 세션은 비구매 세션보다 전체 행동량, 조회 수, 장바구니 수가 많음
# - 평균 조회 상품 수는 3.01 vs 2.97로 거의 차이가 없음
# - 구매 세션의 중앙값 세션 길이는 257초로 비구매 세션 46초보다 훨씬 김
# - 반면 평균 세션 길이는 비구매 세션이 더 길어, 일부 매우 긴 비구매 세션의 영향이 큰 것으로 보임
# - 구매 여부를 구분하는 핵심 차이는 상품 다양성보다 행동 깊이와 장바구니 행동에 더 가까워 보임

In [13]:
# 세션 길이 구간별 구매 세션 비율 확인
# 세션이 얼마나 오래 지속됐는지에 따라 구매가 발생한 세션 비율이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,

            DATE_DIFF(
                'second',
                MIN(event_time),
                MAX(event_time)
            ) AS session_seconds,

            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session
    )

    SELECT
        CASE
            WHEN session_seconds <= 10 THEN '1. <=10s'
            WHEN session_seconds <= 60 THEN '2. 11-60s'
            WHEN session_seconds <= 300 THEN '3. 1-5min'
            WHEN session_seconds <= 600 THEN '4. 5-10min'
            WHEN session_seconds <= 1800 THEN '5. 10-30min'
            ELSE '6. 30min+'
        END AS duration_range,

        COUNT(*) AS sessions,
        SUM(purchased) AS purchase_sessions,

        ROUND(
            SUM(purchased) * 100.0 / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_behavior

    GROUP BY duration_range
    ORDER BY duration_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬──────────┬───────────────────┬───────────────────────┐
│ duration_range │ sessions │ purchase_sessions │ purchase_session_rate │
│    varchar     │  int64   │      int128       │        double         │
├────────────────┼──────────┼───────────────────┼───────────────────────┤
│ 1. <=10s       │  8432097 │              3989 │                  0.05 │
│ 2. 11-60s      │  2791086 │            107637 │                  3.86 │
│ 3. 1-5min      │  5892379 │            667950 │                 11.34 │
│ 4. 5-10min     │  2358460 │            296330 │                 12.56 │
│ 5. 10-30min    │  2020090 │            246499 │                  12.2 │
│ 6. 30min+      │   758844 │             80353 │                 10.59 │
└────────────────┴──────────┴───────────────────┴───────────────────────┘



In [15]:
# 확인 결과
# - 세션이 매우 짧을수록 구매 세션 비율이 매우 낮음
# - 5~10분 구간에서 구매 세션 비율이 12.56%로 가장 높음
# - 10~30분도 12.20%로 비슷한 수준
# - 30분 이상에서는 10.59%로 다시 하락
# - 구매는 너무 짧거나 너무 긴 세션보다 중간 길이의 세션에서 더 자주 발생하는 패턴

In [17]:
# 장바구니 행동 여부에 따라 구매 세션 비율이 얼마나 달라지는지 확인
# 장바구니 행동이 있었던 세션과 없었던 세션의 구매 발생률 차이 확인
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,

            CASE
                WHEN SUM(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) > 0
                THEN 1
                ELSE 0
            END AS has_cart,

            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session
    )

    SELECT
        CASE
            WHEN has_cart = 1 THEN 'cart_session'
            ELSE 'no_cart_session'
        END AS cart_type,

        COUNT(*) AS sessions,
        SUM(purchased) AS purchase_sessions,

        ROUND(
            SUM(purchased) * 100.0 / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_behavior

    GROUP BY cart_type
    ORDER BY cart_type
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬──────────┬───────────────────┬───────────────────────┐
│    cart_type    │ sessions │ purchase_sessions │ purchase_session_rate │
│     varchar     │  int64   │      int128       │        double         │
├─────────────────┼──────────┼───────────────────┼───────────────────────┤
│ cart_session    │  2060635 │            940219 │                 45.63 │
│ no_cart_session │ 20192321 │            462539 │                  2.29 │
└─────────────────┴──────────┴───────────────────┴───────────────────────┘



In [21]:
# 확인 결과
# - 장바구니 행동이 있는 세션의 구매 발생률은 45.63%
# - 장바구니 행동이 없는 세션의 구매 발생률은 2.29%
# - cart 행동은 구매 의도를 강하게 구분하는 행동 신호로 보임
# - 단, cart 없이 purchase가 발생한 세션도 존재하므로 필수 경로로 단정할 수는 없음

In [19]:
# 세션 내 조회 횟수 구간별 구매 세션 비율 확인
# 한 세션에서 조회 행동이 많아질수록 구매 발생률이 어떻게 변하는지 확인
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,

            SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS view_events,

            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session
    )

    SELECT
        CASE
            WHEN view_events = 1 THEN '1. 1 view'
            WHEN view_events <= 3 THEN '2. 2-3 views'
            WHEN view_events <= 5 THEN '3. 4-5 views'
            WHEN view_events <= 10 THEN '4. 6-10 views'
            WHEN view_events <= 20 THEN '5. 11-20 views'
            ELSE '6. 21+ views'
        END AS view_range,

        COUNT(*) AS sessions,
        SUM(purchased) AS purchase_sessions,

        ROUND(
            SUM(purchased) * 100.0 / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_behavior

    GROUP BY view_range
    ORDER BY view_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┬──────────┬───────────────────┬───────────────────────┐
│   view_range   │ sessions │ purchase_sessions │ purchase_session_rate │
│    varchar     │  int64   │      int128       │        double         │
├────────────────┼──────────┼───────────────────┼───────────────────────┤
│ 1. 1 view      │  8543702 │            245694 │                  2.88 │
│ 2. 2-3 views   │  6182114 │            520652 │                  8.42 │
│ 3. 4-5 views   │  2662660 │            227495 │                  8.54 │
│ 4. 6-10 views  │  2734972 │            224335 │                   8.2 │
│ 5. 11-20 views │  1478744 │            121224 │                   8.2 │
│ 6. 21+ views   │   650764 │             63358 │                  9.74 │
└────────────────┴──────────┴───────────────────┴───────────────────────┘



In [24]:
# 확인 결과
# - 조회가 1회뿐인 세션의 구매 세션 비율은 2.88%로 낮음
# - 2회 이상 조회한 세션은 대부분 약 8~10% 수준
# - 조회 횟수가 늘수록 구매율이 계속 증가하는 선형 관계는 관찰되지 않음
# - 단순 조회량보다 cart 여부나 세션 길이 같은 행동 특성이 구매를 더 잘 구분하는 것으로 보임

In [23]:
# 세션에서 조회한 서로 다른 상품 수에 따라 구매 세션 비율 확인
# 한 세션에서 여러 상품을 비교할수록 전체 구매 발생률이 어떻게 달라지는지 확인
duckdb.sql(f"""
    WITH session_behavior AS (
        SELECT
            user_session,

            COUNT(
                DISTINCT CASE
                    WHEN event_type = 'view' THEN product_id
                END
            ) AS viewed_products,

            CASE
                WHEN SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) > 0
                THEN 1
                ELSE 0
            END AS purchased

        FROM read_parquet('{parquet_path}')

        WHERE user_session IS NOT NULL
          AND CAST(event_time AS DATE) <> '2019-11-15'

        GROUP BY user_session
    )

    SELECT
        CASE
            WHEN viewed_products = 1 THEN '1. 1 product'
            WHEN viewed_products <= 3 THEN '2. 2-3 products'
            WHEN viewed_products <= 5 THEN '3. 4-5 products'
            WHEN viewed_products <= 10 THEN '4. 6-10 products'
            ELSE '5. 11+ products'
        END AS product_range,

        COUNT(*) AS sessions,
        SUM(purchased) AS purchase_sessions,

        ROUND(
            SUM(purchased) * 100.0 / COUNT(*),
            2
        ) AS purchase_session_rate

    FROM session_behavior

    WHERE viewed_products > 0

    GROUP BY product_range
    ORDER BY product_range
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬──────────┬───────────────────┬───────────────────────┐
│  product_range   │ sessions │ purchase_sessions │ purchase_session_rate │
│     varchar      │  int64   │      int128       │        double         │
├──────────────────┼──────────┼───────────────────┼───────────────────────┤
│ 1. 1 product     │ 11452448 │            689673 │                  6.02 │
│ 2. 2-3 products  │  5726623 │            399636 │                  6.98 │
│ 3. 4-5 products  │  2181492 │            131475 │                  6.03 │
│ 4. 6-10 products │  1885933 │            110255 │                  5.85 │
│ 5. 11+ products  │   995707 │             64933 │                  6.52 │
└──────────────────┴──────────┴───────────────────┴───────────────────────┘



In [27]:
# 확인 결과
# - 조회 상품 수에 따른 전체 구매 세션 비율은 대부분 6% 안팎으로 비슷함
# - 2~3개 상품을 본 세션이 6.98%로 가장 높지만 차이는 크지 않음
# - 전체 세션 기준으로는 조회 상품 다양성이 구매 여부를 강하게 구분하지 못함

In [29]:
'''
03_cart_abandonment 에서 이미 장바구니까지 간 사람들만 놓고 본 경우 
1개 상품 조회 → 구매전환 49.65%, 11개 이상 조회 → 28.77%
로 꽤 강한 차이가 있었지만 지금 모든 세션을 대상으로 하니 차이가 거의 없음
즉, 전체 방문 단계에서는 상품을 몇 개 봤는지가 구매 여부를 크게 구분하지 않지만, 
이미 장바구니까지 간 고의도 사용자 집단에서는 많은 상품을 비교하는 행동이 구매 이탈과 관련된 패턴을 보인다.
'''

'\n03_cart_abandonment 에서 이미 장바구니까지 간 사람들만 놓고 본 경우 \n1개 상품 조회 → 구매전환 49.65%, 11개 이상 조회 → 28.77%\n로 꽤 강한 차이가 있었지만 지금 모든 세션을 대상으로 하니 차이가 거의 없음\n즉, 전체 방문 단계에서는 상품을 몇 개 봤는지가 구매 여부를 크게 구분하지 않지만, \n이미 장바구니까지 간 고의도 사용자 집단에서는 많은 상품을 비교하는 행동이 구매 이탈과 관련된 패턴을 보인다.\n'